# 07 · SEI 상(相) 매핑 — halo·ring·spot 픽셀 분류 → phase map + 두께

흐름: **입력 → 전처리 → 개요(median/MAX/center) → radial(예상 peak) → Halo → Ring → Spot → Phase map(확정/예상/약함) → 두께**.
각 픽셀에서 radial integration을 해 halo(비정질)·ring(다결정)·spot(단결정)을 보고, 물질과 상을 **확정/예상/약함**으로 판정한다.
로직은 패키지(`fds.classify_pixels` 등), 이 노트북은 **각 셀 파라미터를 노출**해 데이터마다 조절한다.
플롯 텍스트는 ASCII(한글은 마크다운/print만). 판정: **확정**=스팟 인덱싱(격자 자기일관) / **예상**=링 지문 일치 / **약함**=물질이나 상 불명.

## 1) 입력 — 로드 (경로만 바꾸면 어느 데이터든)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"   # ★데이터셋
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN    = 1                       # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT= "1/nm"                 # dm 단위(0.043888 1/nm)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
N_JOBS     = -1                      # 병렬 코어(-1=전부; 32코어면 32)
PHASE_COL  = {"LiF":"#2ca02c","Li2O":"#1f77b4","Li3N":"#9467bd","Li2CO3":"#ff7f0e","Li2S":"#8c564b"}

def _synth(Sy=22,Sx=30,H=88,W=88,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/8); halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def sp(r0,n=6,a=4):
        im=np.zeros((H,W))
        for k in range(n):
            t=2*np.pi*k/n; im+=a*np.exp(-((xx-cx-r0*np.cos(t))**2+(yy-cy-r0*np.sin(t))**2)/(2*1.6**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-3 else (beam+sp(20)+0.5*halo(20) if ix<Sx//2 else beam+1.2*halo(18))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube
if USE_SYNTHETIC: cube=fds.from_array(_synth(),q_per_px=0.02,name="synthetic")
else:
    cube=fds.load(DM4_PATH,Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube,DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=("synthetic" if USE_SYNTHETIC else os.path.splitext(os.path.basename(DM4_PATH))[0])
SAVE_DIR=("nb7_outputs" if USE_SYNTHETIC else os.path.dirname(DM4_PATH)+"/nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,h,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(h); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"| cores:",os.cpu_count(),"->",os.path.abspath(SAVE_DIR))

## 2) 전처리 — 진단(측정 먼저, 필요할 때만 교정)

인공물(중심/wander/타원/defect)은 교정, 신호 평활(블러)은 금지. 여기선 중심과 상태를 진단한다.

In [ ]:
# --- 이 셀 파라미터 ---
HOT_THRESHOLD  = 8.0    # hot/dead 검출 민감도(작을수록 민감). 진짜 스팟이 지워지면 키우기
WANDER_WARN_PX = 1.0    # 이보다 크면 per-position 정렬 권고
ELLIP_WARN     = 0.02   # 타원율 이보다 크면 타원 보정 권고
diag=fds.diagnose_cube(cube,hot_threshold=HOT_THRESHOLD); center=diag["center"]
print("=== 전처리 진단 ===")
print(f"  center           = ({center[0]:.1f},{center[1]:.1f})  (hot-pixel 제거 후 무게중심)")
print(f"  beam wander      = {diag['wander_px']:.2f} px  (>{WANDER_WARN_PX} 이면 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>{100*ELLIP_WARN:.0f}% 면 보정)")
for n in diag["notes"]: print("  -",n)

## 3) 개요 — median / MAX NBD / center

median=비정질 halo가 잘 보임, MAX=다결정 링·스팟이 모여 보임.

In [ ]:
# --- 이 셀 파라미터 ---
CENTER = None       # None=진단값. 수동이면 (cx,cy)
if CENTER is not None: center=CENTER
med=fds.median_pattern(cube); mx=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
fig,ax=plt.subplots(1,2,figsize=(9,4.4))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median NBD (log) - amorphous halo"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX NBD (gamma) - rings/spots"); ax[1].axis("off")
plt.tight_layout(); save(fig,"03_overview"); plt.show()

## 4) Radial integration — 예상되는 물질의 peak 위치 표시

전체 평균/최대 NBD의 방위각 적분에 **후보 5상의 링 위치**를 겹쳐 표시하고, 비정질 **FSDP**(halo 봉우리)를 찾는다.

In [ ]:
# --- 이 셀 파라미터 ---
RING_QBEAM=0.20; RING_QMAX=1.0; RING_NSIG=1.5; RING_TOPN=8   # 링 검출
meanpat=fds.to_pattern(cube)                                    # mean(전체) — halo가 median보다 잘 남음
qd,Id=fds.azimuthal_integrate(meanpat,center,q_per_px=QPP)     # mean -> halo/FSDP
qm,Im=fds.azimuthal_integrate(mx,center,q_per_px=QPP)          # max -> rings
halo_q,halo_conf=fds.find_fsdp(qd,Id,q_lo=max(RING_QBEAM,0.10),q_hi=RING_QMAX)
if not np.isfinite(halo_q): halo_q=0.25                        # 폴백(못 찾으면 ~4A)
rings_q=fds.detect_rings(mx,center,QPP,q_beam=RING_QBEAM,q_max=RING_QMAX,nsig=RING_NSIG,top_n=RING_TOPN)
rings_d=[round(1/q,2) for q in rings_q]
print(f"amorphous FSDP: q={halo_q:.3f} d={1/halo_q:.2f}A conf {halo_conf:.1f}")
print(f"검출 링 d(A) = {rings_d}")
fig,ax=plt.subplots(1,1,figsize=(10,4.2))
ax.semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.9,label="MAX I(q)")
ax.semilogy(qd,np.clip(Id,1e-2,None),"0.5",lw=0.8,label="median I(q)")
for c in CANDIDATES:
    for dd,w in fds.COMPOUND_RINGS[c]:
        ax.axvline(1/dd,color=PHASE_COL[c],ls="--",lw=0.4+0.9*w,alpha=0.5)
for q in rings_q: ax.axvline(q,color="k",ls=":",lw=0.7)
ax.axvline(halo_q,color="r",ls="-",lw=1.2,label=f"FSDP d={1/halo_q:.2f}")
import matplotlib.patches as mp
ax.legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES]+
          [plt.Line2D([],[],color="k",ls=":",label="detected ring"),plt.Line2D([],[],color="r",label="FSDP")],fontsize=7,ncol=2)
ax.set_xlabel("q (1/A)"); ax.set_ylabel("I(q)"); ax.set_title("radial integration + expected phase rings (dashed=candidate d)")
plt.tight_layout(); save(fig,"04_radial_expected"); plt.show()

## 5) Halo 분석 — 물질/진공 판정 + 구조적 halo (진공 대비 cutoff)

**환형 detector + 진공 cutoff.** 진공은 노이즈/artifact 바닥이라, 그 위로 **N σ 이상이면 진짜 산란(물질)**. 두 가지 맵:
- **PLAIN**(halo 밴드 세기): 물질/진공 판정에 쓰지만 **세기 ~ 두께**라 두꺼운 곳이 그냥 밝음(구조 아님).
- **STRUCTURAL**(FSDP *봉우리*): 봉우리 **양옆에 좁은 flank** 두 개를 잡아 그 사이를 **직선으로 fit해 빼고** 봉우리만 남김 → **진짜 비정질 질서**만 밝음.
  두껍기만 하고 봉우리 없는 곳(featureless 빔꼬리)은 직선 위에 놓여 residual~0 → "세기만 높아 물질로 착각"을 배제.
  ★핵심: FSDP가 **빔에 가까운 저-q**(~0.25)면 flank를 **넓히면 안 됨**(아래 flank가 빔꼬리에 빠져 fit이 망가짐).
  flank는 **좁게**, 그리고 `beam_cut`으로 아래 flank가 빔 안으로 못 들어가게 막는다(beam guard).
halo는 비정질이라 상 이름은 안 붙임(2Å 축퇴). LiF 여부는 §6 ring이 답.

In [ ]:
# --- 이 셀 파라미터 ---
VAC_PCTL      = 15          # 엄격 기준진공 = 총세기 하위 X% (detector cutoff 기준). 물질이 화면을 많이 채우면 낮추기
HALO_BAND     = (0.15,0.60) # 물질 판정용 broad halo 밴드(1/A) — 이 구간 산란이 진공 대비 유의하면 물질
HALO_SIGMA    = 3.0         # 진공 대비 이 σ 이상 = 진짜 물질(cutoff). ★물질이 과하게 잡히면 키우기(4,5)
HALO_STRONG_S = 8.0         # 이 σ 이상 = 확정물질(강 halo), 그 사이는 예상물질
HALO_DQ       = 0.04        # 개별 halo 반경 detector 반폭(그림용, 1/A)
FSDP_RANGE    = (0.20,0.35) # 구조적 halo용 FSDP 봉우리를 찾을 q 범위(1/A) — NBD 감마·bandpass로 보이는 ~0.25 부근
# 저-q FSDP는 flank를 '좁게'(넓히면 아래 flank가 빔꼬리에 빠짐). beam guard가 아래 flank를 빔 밖으로 강제.
STRUCT_DQ     = 0.05        # 구조적 halo 밴드 반폭(1/A) — 봉우리 폭(FWHM 절반)에 맞춰 좁게
STRUCT_FGAP   = 0.07        # flank를 봉우리 중심에서 이만큼 밖에(=봉우리 바로 옆). ★좁게 유지
STRUCT_FW     = 0.03        # flank 창 폭(1/A)
STRUCT_BEAM   = 0.15        # 이 q 아래는 빔 영역 — 아래 flank가 여기로 못 내려가게 clamp(beam guard)
NBIN          = 200
# 1) 픽셀 radial stack 1회 + 엄격 진공
q_stack,prof=fds.radial_stack(cube,center,QPP,q_max=1.2,nbin=NBIN,n_jobs=N_JOBS)
vac_ref=fds.strict_vacuum_mask(cube,center=center,q_per_px=QPP,pctl=VAC_PCTL)
# 2) 물질 판정 = broad halo 밴드 전체를 진공 대비(plain 환형; 정확한 peak 불필요, robust)
q0b=0.5*(HALO_BAND[0]+HALO_BAND[1]); dqb=0.5*(HALO_BAND[1]-HALO_BAND[0])
halo_sig,_=fds.detector_map(cube,center,QPP,q0b,dq=dqb,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))
material=halo_sig>HALO_SIGMA
halo_tier=np.where(~material,0,np.where(halo_sig>=HALO_STRONG_S,3,2))
print(f"물질 {100*material.mean():.0f}% (halo밴드 {HALO_BAND}>{HALO_SIGMA}σ) | 확정물질(>{HALO_STRONG_S}σ) {int((halo_tier==3).sum())}px | 예상물질 {int((halo_tier==2).sum())}px | 진공 {int((~material).sum())}px")
# 물질 평균 radial (FSDP·secondary halo 찾기용)
med_rad=prof[material.ravel()].mean(0) if material.any() else prof.mean(0)
# robust FSDP q (물질평균 봉우리; §4 median-over-all은 nan 가능)
_bl=np.interp(q_stack,[FSDP_RANGE[0]-0.03,FSDP_RANGE[1]+0.03],[med_rad[np.argmin(np.abs(q_stack-(FSDP_RANGE[0]-0.03)))],med_rad[np.argmin(np.abs(q_stack-(FSDP_RANGE[1]+0.03)))]])
_ssel=(q_stack>=FSDP_RANGE[0])&(q_stack<=FSDP_RANGE[1])
fsdp_q=float(q_stack[_ssel][np.argmax((med_rad-_bl)[_ssel])]) if _ssel.any() else 0.25
# halo 프로브: FSDP(자동) + 고정 secondary. 각 반경에 PLAIN(세기) & STRUCTURAL(봉우리) 둘 다
halo_probes=[p for p in ([round(fsdp_q,3),0.30,0.40,0.80]) if 0.12<p<1.0]
plain_maps=[fds.detector_map(cube,center,QPP,p,dq=HALO_DQ,flank=None,vacuum_mask=vac_ref,_stack=(q_stack,prof))[0] for p in halo_probes]
struct_maps=[fds.structural_halo_map(cube,center,QPP,p,dq=STRUCT_DQ,flank_gap=STRUCT_FGAP,flank_w=STRUCT_FW,beam_cut=STRUCT_BEAM,vacuum_mask=vac_ref,_stack=(q_stack,prof))[0] for p in halo_probes]
struct_sig=struct_maps[0]
print(f"FSDP q={fsdp_q:.3f} (d={1/fsdp_q:.2f}A) | halo 프로브 q =",[round(p,3) for p in halo_probes])
print("  PLAIN  유의(>3σ) % (세기=두께):",[round(100*(sg>HALO_SIGMA).mean(),1) for sg in plain_maps])
print("  STRUCT 유의(>3σ) % (봉우리=구조):",[round(100*(sg>3).mean(),1) for sg in struct_maps]," ← 이게 진짜 halo 봉우리")
# --- 그림 A: 대표 halo NBD | PLAIN(세기=두께) | STRUCTURAL(FSDP 봉우리) | 물질 마스크 ---
iy,ix=np.unravel_index(int(np.argmax(np.where(material,halo_sig,-np.inf))),scan)
fig,ax=plt.subplots(1,4,figsize=(18,4.2))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
ax[0].add_patch(plt.Circle(center,fsdp_q/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
ax[0].set_title(f"representative HALO pixel ({iy},{ix})\nFSDP d={1/fsdp_q:.2f}A (cyan)"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,halo_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("PLAIN halo-band (intensity ~ thickness)"); ax[1].axis("off")
im=ax[2].imshow(np.where(material,struct_sig,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[2],fraction=0.046)
ax[2].set_title(f"STRUCTURAL halo @FSDP {fsdp_q:.2f} (bump only)"); ax[2].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmH=ListedColormap(["#000000","#4575b4","#2ca02c"]); nmH=BoundaryNorm([-.5,.5,2.5,3.5],3)
ax[3].imshow(halo_tier,cmap=cmH,norm=nmH); ax[3].set_title(f"material (>{HALO_SIGMA}s): vacuum/weak/strong"); ax[3].axis("off")
plt.tight_layout(); save(fig,"05_halo_material"); plt.show()
# --- 그림 B: 각 halo 반경 PLAIN(위) vs STRUCTURAL(아래) — 0.3/0.4/0.8이 진짜 봉우리인지 ---
nH=len(halo_probes)
fig,ax=plt.subplots(2,nH,figsize=(3.4*nH,7.0)); ax=np.atleast_2d(ax)
for j,(p,pl,st) in enumerate(zip(halo_probes,plain_maps,struct_maps)):
    im=ax[0,j].imshow(np.where(material,pl,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[0,j],fraction=0.046)
    ax[0,j].set_title(f"PLAIN q={p:.2f} (d={1/p:.2f}A)",fontsize=9); ax[0,j].axis("off")
    im=ax[1,j].imshow(np.where(material,st,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1,j],fraction=0.046)
    ax[1,j].set_title(f"STRUCTURAL q={p:.2f} (bump)",fontsize=9); ax[1,j].axis("off")
fig.suptitle("per-halo-radius: PLAIN (intensity/thickness) vs STRUCTURAL (real bump)")
plt.tight_layout(); save(fig,"05_halo_peaks"); plt.show()
# --- 그림 C: flank 구성 그 자체 — 각 halo 위치에서 flank 2개로 직선 배경을 fit하고
#     그 위 '봉우리'를 칠해 보여준다(구조적 halo map이 실제로 재는 그 값). material-vacuum(단순 빼기)은
#     물질 자기 배경이 안 빠져 봉우리를 숨기므로 쓰지 않는다. ★설명할 때 그린 flank-봉우리 그림의 실제 데이터판.
vac_rad=prof[vac_ref.ravel()].mean(0)
def _flank_demo(qq,p1d,q0,dq=STRUCT_DQ,fgap=STRUCT_FGAP,fw=STRUCT_FW,beam=STRUCT_BEAM):
    band=(qq>=q0-dq)&(qq<=q0+dq)
    lo0=q0-fgap-fw
    lo_s,lo_e=(beam,beam+fw) if lo0<beam else (lo0,q0-fgap)   # beam guard
    fl=((qq>=lo_s)&(qq<=lo_e))|((qq>=q0+fgap)&(qq<=q0+fgap+fw))
    qf=qq[fl]; qfm=qf.mean(); d=qf-qfm; den=float((d**2).sum()) or 1.0
    slope=(p1d[fl]*d).sum()/den; icpt=p1d[fl].mean()-slope*qfm
    return band,fl,(slope*qq+icpt)
fig,ax=plt.subplots(1,1+nH,figsize=(3.4*(1+nH),3.9))
ax=np.atleast_1d(ax)
# 패널0: 물질 vs 진공 log I(q) (물질이 진공 위에 있다는 맥락만; 빼기 아님)
ax[0].semilogy(q_stack,np.clip(med_rad,1e-3,None),"k-",lw=1.1,label="material mean")
ax[0].semilogy(q_stack,np.clip(vac_rad,1e-3,None),"0.6",lw=1.0,label="vacuum mean")
for p in halo_probes: ax[0].axvline(p,color="r",ls=":",lw=0.8)
ax[0].set_xlim(0.12,1.1); ax[0].set_xlabel("q (1/A)"); ax[0].set_title("I(q): material vs vacuum (context)"); ax[0].legend(fontsize=8)
# 패널1..: 각 프로브에서 flank 직선 배경 + 그 위 봉우리(칠)
for a,p in zip(ax[1:],halo_probes):
    band,fl,base=_flank_demo(q_stack,med_rad,p)
    win=(q_stack>=p-3.2*STRUCT_FGAP)&(q_stack<=p+3.2*STRUCT_FGAP)
    a.plot(q_stack[win],med_rad[win],"k-",lw=1.3,label="material mean")
    a.plot(q_stack[win],base[win],"r--",lw=1.1,label="flank baseline")
    # flank 창(회색), 봉우리 밴드(주황 칠: profile-baseline>0)
    for lab,mask,col in [("flank",fl,"0.7")]:
        qs=q_stack[mask]
        if qs.size:
            a.axvspan(qs.min(),qs[qs<p].max() if (qs<p).any() else qs.min(),color=col,alpha=0.25)
            if (qs>p).any(): a.axvspan(qs[qs>p].min(),qs.max(),color=col,alpha=0.25)
    bump=np.clip(med_rad-base,0,None)
    a.fill_between(q_stack,base,med_rad,where=band&(med_rad>base),color="tab:orange",alpha=0.65,label="bump (STRUCTURAL)")
    a.axvline(p,color="r",ls=":",lw=0.8)
    a.set_xlim(p-3.2*STRUCT_FGAP,p+3.2*STRUCT_FGAP)
    ymax=float(med_rad[win].max())*1.08; ymin=float(min(base[band].min(),med_rad[win].min()))*0.96
    a.set_ylim(ymin,ymax); a.set_xlabel("q (1/A)")
    sf=(struct_maps[halo_probes.index(p)]>3).mean()
    a.set_title(f"flank @q={p:.2f} (d={1/p:.2f}A)\nbump area={bump[band].sum():.0f} | struct>3s {100*sf:.0f}%",fontsize=9)
    a.legend(fontsize=7,loc="upper right")
fig.suptitle("flank construction: baseline from two flanks, orange bump above it = the STRUCTURAL halo the map measures")
plt.tight_layout(); save(fig,"05_halo_profile"); plt.show()
# --- 그림 D: 각 halo 반경에서 '강한 신호 픽셀'의 평균 NBD — 뿌연 링(halo)인가 스팟(결정질)인가 ---
STRONG_PCTL=90   # 각 반경 STRUCTURAL 맵 상위 %를 '강한 신호'로 골라 그 픽셀들의 NBD 평균
fig,ax=plt.subplots(1,nH,figsize=(3.6*nH,3.8)); ax=np.atleast_1d(ax)
for a,p,st in zip(ax,halo_probes,struct_maps):
    thr=np.nanpercentile(st[material],STRONG_PCTL) if material.any() else np.inf
    strong=material&(st>=thr)&(st>0)
    n=int(strong.sum())
    if n>=3:
        mp=fds.average_pattern(cube,strong)
        a.imshow((np.clip(mp,0,None)/np.max(mp))**0.3,cmap="magma"); a.plot(*center,"c+",ms=7)
        a.add_patch(plt.Circle(center,p/QPP,fill=False,ec="cyan",ls="--",lw=1.0))
        a.set_title(f"q={p:.2f} (d={1/p:.2f}A)\nmean NBD of {n} strong px",fontsize=9)
    else:
        a.set_title(f"q={p:.2f}: strong px<3",fontsize=9)
    a.axis("off")
fig.suptitle("mean NBD at STRONG STRUCTURAL pixels per halo radius: diffuse RING=amorphous halo, SPOTS=crystalline")
plt.tight_layout(); save(fig,"05_halo_meanNBD"); plt.show()
save_csv("05_halo",["halo_q","halo_d","plain_frac>3s","struct_frac>3s"],
         [[f"{p:.4f}",f"{1/p:.3f}",f"{(pl>HALO_SIGMA).mean():.4f}",f"{(st>3).mean():.4f}"] for p,pl,st in zip(halo_probes,plain_maps,struct_maps)]
         +[["material_band","",f"{material.mean():.4f}",""]])

## 6) Ring 분석 — 다결정 링 detector + 진공/배경 대비 (상 예상)

**각 상의 sharp 결정질 링**에 flank-빼기 detector를 놓는다(broad halo는 자동 무시). 링 세기를 **비정질 배경(물질) 대비 z-score**로
보면 진짜 sharp 링만 튄다. 한 영역에서 **그 상의 링이 여러 개**(가중 비율) 유의하면 그 상 **예상**. (2.0·2.8 공유 링은 그 상의
*다른* 링들도 함께 있어야 높음 → 링 세트로 판정.) 상별 evidence 맵 + ring-예상 상 맵 + 대표 ring 픽셀.

In [ ]:
# --- 이 셀 파라미터 ---
RING_DQ       = 0.03   # 링 detector 반폭(1/A) ~ 링 폭
RING_SIGMA    = 4.0    # 비정질 배경 대비 이 σ 이상 = 그 링 유의. ★잡링 많으면 키우기
RING_EVID_MIN = 0.40   # 그 상 링의 가중 비율 이 이상 유의 → 예상. 낮추면 예상↑
RING_QLO      = 0.25   # 링 매칭 하한(1/A, FSDP/halo 제외)
ring_ev=fds.ring_phase_evidence(cube,center=center,q_per_px=QPP,candidates=CANDIDATES,ref_mask=material,
        dq=RING_DQ,flank=2.0,ring_sigma=RING_SIGMA,q_lo=RING_QLO,q_max=1.15,_stack=(q_stack,prof))
ev_stack=np.stack([ring_ev[c]["evidence"] for c in CANDIDATES],0)      # (nphase,Sy,Sx)
ring_best=np.argmax(ev_stack,0); ring_best_e=np.max(ev_stack,0)
ring_pred=material&(ring_best_e>=RING_EVID_MIN)                        # 링으로 상 예상된 픽셀
ring_pred_idx=np.where(ring_pred,ring_best,-1)
print("=== ring detector 상 예상 (비정질 배경 대비) ===")
for k,c in enumerate(CANDIDATES):
    npx=int(((ring_best==k)&ring_pred).sum())
    if npx: print(f"  {c:7s}: {npx} px  (평균 evidence {ev_stack[k][(ring_best==k)&ring_pred].mean():.2f}, 링 최대 {int(ring_ev[c]['n_sig'].max())}개 유의)")
print(f"  ring 예상 총 {int(ring_pred.sum())} px / 물질 {int(material.sum())} px")
# 대표 ring 픽셀 = 최고 evidence
iy,ix=np.unravel_index(int(np.argmax(np.where(material,ring_best_e,-1))),scan); kbest=int(ring_best[iy,ix])
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="magma"); ax[0].plot(*center,"c+",ms=8)
for dd,w in fds.COMPOUND_RINGS[CANDIDATES[kbest]]:
    if RING_QLO<=1/dd<=1.15: ax[0].add_patch(plt.Circle(center,(1/dd)/QPP,fill=False,ec=PHASE_COL[CANDIDATES[kbest]],ls="--",lw=0.4+0.9*w,alpha=0.8))
ax[0].set_title(f"representative RING pixel ({iy},{ix})\nbest: {CANDIDATES[kbest]} (evid {ring_best_e[iy,ix]:.2f})"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,ring_best_e,np.nan),cmap="inferno",vmin=0,vmax=1); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("best-phase ring evidence (0..1)"); ax[1].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
import matplotlib.patches as mp
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
ax[2].imshow(np.where(ring_pred,ring_pred_idx,np.nan),cmap=cmP,norm=nmP)
ax[2].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
ax[2].set_title(f"ring-predicted phase (evid>={RING_EVID_MIN})"); ax[2].axis("off")
plt.tight_layout(); save(fig,"06_ring"); plt.show()
save_csv("06_ring",["phase","predicted_px","mean_evidence"],
         [[c,int(((ring_best==k)&ring_pred).sum()),f"{(ev_stack[k][ring_pred].mean() if ring_pred.any() else 0):.3f}"] for k,c in enumerate(CANDIDATES)])

## 7) Spot 분석 — ring 영역 안에서만 클러스터 인덱싱 (확정)

**ring 예상 영역(그리고 spottiness 높은 곳)만** 대상으로 한다 — 노이즈에서 수천 개 잡던 문제를 없앤다. 그 영역을
**연결성분(grain)으로 묶어** 각 grain의 NBD를 평균내 스팟을 검출·인덱싱: 한 상으로 **≥4개 자기일관(|g|+각도)=확정**,
갈리거나 2개=예상. 대표 spot 픽셀 NBD(검출 스팟), spottiness 맵, grain별 판정 오버레이.

In [ ]:
# --- 이 셀 파라미터 ---
SPOT_REGION_PCTL = 90    # ring 예상이 적을 때 보강: 물질 내 spottiness 상위 %도 결정질 후보에 포함
SPOT_MIN_GRAIN   = 2     # 최소 grain 크기(px)
SPOT_NMAD        = 8.0   # grain 평균 DP 스팟 문턱(median+NMAD*MAD). 스팟 너무 많으면 키우기
from scipy.ndimage import label as _label
spotmap=fds.crystallinity_map(cube,center=center,q_per_px=QPP,n_jobs=N_JOBS)   # 방위각 spottiness
# 결정질 후보 영역 = ring 예상 ∪ (물질 내 spottiness 상위)
region=ring_pred | (material & (spotmap>np.nanpercentile(spotmap[material],SPOT_REGION_PCTL)))
labels,ng=_label(region)
print(f"=== spot: 결정질 영역 {int(region.sum())}px → grain {ng}개 (ring 예상 {int(ring_pred.sum())}px 포함) ===")
spot_tier=np.zeros(scan,int); spot_phase=np.full(scan,-1,int); confirmed=set(); grain_rows=[]
for g in range(1,ng+1):
    m=labels==g
    if m.sum()<SPOT_MIN_GRAIN: continue
    pat=fds.average_pattern(cube,m)
    sp=fds.detect_spots(pat,center,QPP,n_mad=SPOT_NMAD,min_dist=3,tophat=11,q_max=1.15)
    gs=fds.spots_to_gvectors(sp,center,QPP)
    best=None
    if len(gs)>=2: best,_=fds.index_pattern(gs,candidates=CANDIDATES,tol_g=0.03,tol_ang=6.0,min_score=0.6,min_complete=0.6,confirm_min_spots=4)
    if best is None: continue
    ph=best["phase"]; ki=CANDIDATES.index(ph)
    if best["indexed"]: spot_tier[m]=3; spot_phase[m]=ki; confirmed.add(ph); tag="[확정]"
    elif best["n_matched"]>=2: spot_tier[m]=np.maximum(spot_tier[m],2); spot_phase[m]=np.where(spot_phase[m]<0,ki,spot_phase[m]); tag="[예상]"
    else: tag="[약함]"
    grain_rows.append([g,int(m.sum()),len(gs),ph,int(best["indexed"]),str(best["zone"]),best["n_matched"],f"{best['completeness']:.2f}"])
    print(f"  grain{g:2d} {int(m.sum()):3d}px spots{len(gs):2d}: {tag} {ph:6s} zone{str(best['zone']):12s} m{best['n_matched']}/{best['n_total']} comp{best['completeness']:.2f}")
print(f"  => 인덱싱 확정 상: {sorted(confirmed) or '없음'}")
# 대표 spot 픽셀 = spottiness 최대(물질 내)
iy,ix=np.unravel_index(int(np.argmax(np.where(material,spotmap,-np.inf))),scan)
fig,ax=plt.subplots(1,3,figsize=(14,4.3))
sel=np.zeros(scan,bool); sel[iy,ix]=True; pat=fds.average_pattern(cube,sel)
sp=fds.detect_spots(pat,center,QPP,n_mad=SPOT_NMAD,min_dist=3,tophat=11,q_max=1.15)
ax[0].imshow((np.clip(pat,0,None)/np.max(pat))**0.3,cmap="gray"); ax[0].plot(*center,"c+",ms=8)
ax[0].scatter([s[0] for s in sp],[s[1] for s in sp],s=30,facecolors="none",edgecolors="yellow",lw=0.8)
ax[0].set_title(f"representative SPOT pixel ({iy},{ix})\n{len(sp)} spots"); ax[0].axis("off")
im=ax[1].imshow(np.where(material,spotmap,np.nan),cmap="inferno"); plt.colorbar(im,ax=ax[1],fraction=0.046)
ax[1].set_title("spottiness (azimuthal variance)"); ax[1].axis("off")
from matplotlib.colors import ListedColormap,BoundaryNorm
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
ax[2].imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
ax[2].imshow(np.where(spot_tier==3,spot_phase,np.nan),cmap=cmP,norm=nmP)                # 확정 grain(상 색)
ax[2].imshow(np.where(spot_tier==2,1.0,np.nan),cmap="autumn",vmin=0,vmax=2,alpha=0.7)   # 예상 grain
ax[2].set_title("spot grains: color=indexed phase, orange=predict"); ax[2].axis("off")
plt.tight_layout(); save(fig,"07_spot"); plt.show()
if grain_rows: save_csv("07_spot_grains",["grain","size","n_spots","phase","indexed","zone","n_matched","completeness"],grain_rows)

## 8) Phase mapping — 확정 / 예상 / 약함 (종합)

halo/ring/spot 종합: **확정**=spot 인덱싱(§7) · **예상**=ring 지문(§6) · **약함**=물질이나 상 불명(halo만) · **없음**=진공.
확정 > 예상 > 약함 우선. 확정/예상/약함 맵을 상별 색으로.

In [ ]:
# 종합: 확정(spot)=3 > 예상(ring)=2 > 약함(물질)=1 > 진공=0
tier=np.where(~material,0,1)                              # 물질=약함 기본
phase_idx=np.full(scan,-1,int)
pm=ring_pred&(ring_pred_idx>=0)                           # ring 예상
tier[pm]=2; phase_idx[pm]=ring_pred_idx[pm]
cm=spot_tier==3                                           # spot 확정(최우선)
tier[cm]=3; phase_idx[cm]=spot_phase[cm]
pm2=(spot_tier==2)&(tier<2)                               # spot 예상(ring 없던 곳)
tier[pm2]=2; phase_idx[pm2]=spot_phase[pm2]
print("=== 픽셀 종합 판정 ===")
for t,name in [(3,"확정"),(2,"예상"),(1,"약함"),(0,"없음")]: print(f"  {name}: {int((tier==t).sum())} px",end="")
print()
for k,c in enumerate(CANDIDATES):
    cf=int(((phase_idx==k)&(tier==3)).sum()); pr=int(((phase_idx==k)&(tier==2)).sum())
    if cf or pr: print(f"    {c:7s}: 확정 {cf} px, 예상 {pr} px")
conf_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==3)).any()})
pred_ph=sorted({CANDIDATES[k] for k in range(len(CANDIDATES)) if ((phase_idx==k)&(tier==2)).any()})
print(f"  => 확정 상: {conf_ph or '없음'} | 예상 상: {pred_ph or '없음'}")
from matplotlib.colors import ListedColormap,BoundaryNorm
import matplotlib.patches as mp
cmP=ListedColormap([PHASE_COL[c] for c in CANDIDATES]); nmP=BoundaryNorm(np.arange(-.5,len(CANDIDATES)+.5),len(CANDIDATES))
def layer(t): return np.where((tier==t)&(phase_idx>=0),phase_idx,np.nan)
fig,ax=plt.subplots(1,3,figsize=(15,4.4))
for a,(t,ttl) in zip(ax,[(3,"CONFIRMED (indexed)"),(2,"PREDICTED (ring)"),(1,"WEAK (material, no phase)")]):
    a.imshow(np.where(material,0.12,np.nan),cmap="Greys",vmin=0,vmax=1)
    if t==1: a.imshow(np.where(tier==1,1.0,np.nan),cmap="Greys",vmin=0,vmax=1.4)
    else: a.imshow(layer(t),cmap=cmP,norm=nmP)
    a.set_title(f"{ttl}: {int((tier==t).sum())} px"); a.axis("off")
ax[0].legend(handles=[mp.Patch(color=PHASE_COL[c],label=c) for c in CANDIDATES],fontsize=7,loc="lower right",framealpha=.6)
fig.suptitle("PHASE MAP - CONFIRMED / PREDICTED / WEAK (color = phase)")
plt.tight_layout(); save(fig,"08_phase_map"); plt.show()
save_csv("08_phase_tiers",["phase","confirmed_px","predicted_px"],
         [[c,int(((phase_idx==CANDIDATES.index(c))&(tier==3)).sum()),int(((phase_idx==CANDIDATES.index(c))&(tier==2)).sum())] for c in CANDIDATES])

## 9) 두께 계산 — t/lambda = ln(I_total / I_beam)

앞에서 **물질/진공을 이미 판정**했으니 이를 활용한다. 진공(=물질 없는 곳)을 기준으로 dark를 자기교정하고 영점을 맞춘 **상대 두께**.
낮음=얇음(표면/껍데기), 높음=두꺼움. (수렴빔·각도분리라 절대 nm은 λ 필요 → 상대값만 신뢰.)

In [ ]:
# --- 이 셀 파라미터 ---
BEAM_RADIUS_PX=None    # 직접빔 디스크 반경(px). None=자동(~det/20)
VAC_PCTL=15            # 총세기 하위 X% = 엄격 기준진공(dark/영점용, 확실히 빈 곳). 물질이 많으면 낮추기
_flat=cube._flat_patterns(); _tot=np.asarray(_flat,float).reshape(_flat.shape[0],-1).sum(1)
vac_ref=np.asarray(_tot<=np.percentile(_tot,VAC_PCTL),bool).reshape(scan)   # 엄격 기준진공(오염 방지)
tmap,tex=fds.thickness_map(cube,center=center,beam_radius=BEAM_RADIUS_PX,vacuum_mask=vac_ref,return_extras=True)
tin=tmap[material]; tin=tin[np.isfinite(tin)]; tvac=tmap[vac_ref]; tvac=tvac[np.isfinite(tvac)]
print(f"dark 자기교정 D={tex['dark']:.3g} | 영점 offset={tex['offset']:.3g} | beam_r={tex['beam_radius']:.1f}px")
print(f"진공 t/lambda mean {tvac.mean():+.3f} (≈0 정상) | 물질 t/lambda mean {np.mean(tin):.2f} (5~95%: {np.percentile(tin,5):.2f}~{np.percentile(tin,95):.2f})")
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
im=ax[0].imshow(np.where(material,tmap,np.nan),cmap="viridis"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[0].set_title("relative thickness t/lambda (vacuum-referenced)"); ax[0].axis("off")
ax[1].hist(tin,bins=50,color="0.4",label="material"); ax[1].axvline(0,color="r",ls="--",lw=1,label="vacuum (0)")
ax[1].set_xlabel("t/lambda"); ax[1].set_ylabel("# positions"); ax[1].set_title("thickness distribution (thin <-> thick)"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"09_thickness"); plt.show()
save_csv("09_thickness_stats",["metric","value"],
         [["dark",f"{tex['dark']:.4g}"],["vac_mean",f"{tvac.mean():.4f}"],["mat_mean",f"{np.mean(tin):.4f}"],
          ["mat_p05",f"{np.percentile(tin,5):.4f}"],["mat_p95",f"{np.percentile(tin,95):.4f}"],["material_frac",f"{material.mean():.4f}"]])
print("[정리] 확정=스팟 인덱싱된 상 / 예상=링 지문 / 약함=비정질·불명. 두께는 진공 대비 상대값.")